In [36]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import SystemMessage, HumanMessage

In [37]:
from typing import Literal
from pydantic import BaseModel, Field


class IntentStructure(BaseModel):
    intent: Literal["Coding", "Summarization", "Creative"] = Field(
        ...,
        description="Intent of the query"
    )


In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key="REDACTED")

In [40]:
Intent_llm = model.with_structured_output(IntentStructure)

In [53]:
query = f"""
       Write a python code to take two numbers as a input agd give addition as their output
"""

In [54]:
messages = [
        SystemMessage(content="""You are an intent classifier.

Classify the user's query.

Possible intents:

coding
creative
summarization"""),
        HumanMessage(content=query)
        ]


In [55]:
response = Intent_llm.invoke(messages)

In [56]:
response

IntentStructure(intent='Coding')

In [44]:
from typing import TypedDict, Dict
MODEL_REGISTRY: Dict[str, Dict] = {
"coding": {
"provider": "OpenRouter",
"model": "deepseek/deepseek-chat-v3-0324",
"description": "Optimized for programming, debugging, and code generation.",
},
"creative": {
"provider": "OpenRouter",
"model": "google/gemma-3-12b-it",
"description": "Excellent for storytelling, blogs, emails, and creative writing.",
},
"summarization": {
"provider": "OpenRouter",
"model": "qwen/qwen3-8b",
"description": "Optimized for summarization and long-context understanding.",
},
}

In [45]:
def model_selector(intent):
    model_info = MODEL_REGISTRY.get(intent.lower())

    if model_info is None:
        raise ValueError(f"Unknown intent: {intent}")

    print("MODEL SELECTION NODE")
    print(f"Detected Intent : {intent}")
    print(f"Provider        : {model_info['provider']}")
    print(f"Selected Model  : {model_info['model']}")

    return model_info

In [57]:
model_info = model_selector(response.intent)

MODEL SELECTION NODE
Detected Intent : Coding
Provider        : OpenRouter
Selected Model  : deepseek/deepseek-chat-v3-0324


In [47]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


def ask_model(query: str, model: str) -> str:
    """
    Send a query to an OpenRouter model.

    Args:
        query (str): User query.
        model (str): OpenRouter model name.

    Returns:
        str: Model response.
    """

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:8000",   # Optional
        "X-OpenRouter-Title": "AI Router",         # Optional
    }

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ]
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        raise Exception(
            f"Error {response.status_code}: {response.text}"
        )

    result = response.json()

    return result["choices"][0]["message"]["content"]






In [58]:
# --------------------------
result = ask_model(query,model_info['model'])

In [ ]:
print(result)

'# Python Program to Add Two Numbers\n\n```python\n# Take two numbers as input from the user\nnum1 = float(input("Enter first number: "))\nnum2 = float(input("Enter second number: "))\n\n# Add the two numbers\nsum = num1 + num2\n\n# Display the result\nprint("The sum of", num1, "and", num2, "is:", sum)\n```\n\n### Explanation:\n1. The `input()` function is used to take user input, which is by default a string.\n2. `float()` converts the input string to a floating-point number (to handle both integers and decimals).\n3. The two numbers are added and stored in the variable `sum`.\n4. Finally, the result is printed using the `print()` function.\n\n### Example Usage:\n```\nEnter first number: 5\nEnter second number: 3.5\nThe sum of 5.0 and 3.5 is: 8.5\n```\n\nNote: If you specifically want to work with integers only, you can use `int()` instead of `float()`.'